# 09 — Entropic Regularization

Reward the transport plan a little for being *spread out*, and the problem turns smooth, strongly convex, and fast. At its heart sits one object: the Gibbs kernel.

**Prerequisites:** `08_kantorovich_duality`.

**What you'll be able to do**
- Write the entropic optimal-transport objective and explain why the entropy bonus makes it strongly convex.
- Build the Gibbs kernel $K = e^{-C/\varepsilon}$ and watch $\varepsilon$ tune it from sharp to flat.
- Predict the two limits: $\varepsilon \to 0$ recovers the LP, $\varepsilon \to \infty$ gives the independent coupling.

The exact LP is beautiful but slow — $O(n^3)$, hopeless at data scale. The change that unlocked modern optimal transport is small and a little mischievous: give the plan a bonus for being spread out. This entropic term makes the objective smooth and strongly convex, and — as the next notebook shows — solvable in a handful of matrix multiplications. Everything turns on one object, the **Gibbs kernel**, which we meet here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost
from qot_course.transport.sinkhorn import gibbs_kernel

np.random.seed(42)
viz.use_course_style()

## 1. The entropic objective and the Gibbs kernel

Cuturi (2013) added a small entropy bonus to the transport cost:

$$ P^\star_\varepsilon = \arg\min_{P \in T(a, b)}\ \langle C, P\rangle - \varepsilon\, H(P), \qquad H(P) = -\sum_{i, j} P_{ij}\log P_{ij}. $$

Three consequences follow:

- **Strong convexity.** $H$ is strictly concave, so $-\varepsilon H$ is strictly convex and the minimiser is *unique* and varies smoothly with the data.
- **Multiplicative structure.** The optimality conditions force $P^\star_\varepsilon$ to factor as $u_i\, K_{ij}\, v_j$ with the **Gibbs kernel** $K_{ij} = e^{-C_{ij}/\varepsilon}$.
- **Cheap iterations.** The scalings $u, v$ are found by alternating matrix–vector products — no LP solver (the algorithm of the next notebook).

The kernel $K$ is the whole story's centre of gravity, and $\varepsilon$ is its dial. Let's build it and turn the dial.

In [ ]:
positions = np.linspace(0.0, 9.0, 16)
C = squared_euclidean_cost(positions, positions)

epsilons = [0.1, 1.0, 10.0]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
for ax, eps in zip(axes, epsilons):
    K = gibbs_kernel(C, eps)
    im = ax.imshow(K / K.max(), cmap=viz.CMAP_PLAN, origin="lower")  # normalise for display
    ax.set_title(f"eps = {eps}", pad=8)
    ax.set_xlabel("position j")
    ax.grid(False)
    if ax is axes[0]:
        ax.set_ylabel("position i")
    fig.colorbar(im, ax=ax, shrink=0.82)
fig.suptitle("Gibbs kernel K = exp(-C / eps): sharp near-diagonal (small eps) to flat (large eps)",
             fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

**Read the three heatmaps.** At $\varepsilon = 0.1$ the kernel is a thin bright diagonal — it strongly prefers short moves, so the eventual plan will hug the cheapest routes (close to the LP). At $\varepsilon = 1$ the diagonal broadens. At $\varepsilon = 10$ the kernel is nearly uniform: the cost barely registers, and the plan it produces will spread toward the independent coupling. The single dial $\varepsilon$ slides continuously between *sharp transport* and *blind spreading*.

## 2. What the entropy bonus rewards

The bonus $-\varepsilon H(P)$ pulls the plan toward **high entropy**, that is toward being spread out. The most spread-out coupling with the right marginals is the **independent coupling** $a\,b^\top$; the most concentrated is a sharp, permutation-like plan. Comparing their entropies shows what the dial is trading.

In [ ]:
def coupling_entropy(P):
    """Shannon entropy of a coupling, H(P) = -sum P log P (nats)."""
    nonzero = P[P > 0]
    return float(-np.sum(nonzero * np.log(nonzero)))

a = np.full(4, 1 / 4)
b = np.full(4, 1 / 4)
independent = np.outer(a, b)   # most spread-out coupling in T(a, b)
sharp = np.diag(a)             # a concentrated (permutation-like) coupling in T(a, b)

print(f"H(independent coupling a b^T) = {coupling_entropy(independent):.4f} nats   (maximal spread)")
print(f"H(sharp diagonal coupling)    = {coupling_entropy(sharp):.4f} nats   (concentrated)")
print()
print("As eps -> 0   the transport cost dominates  -> sharp plan (the LP optimum).")
print("As eps -> inf the entropy bonus dominates   -> independent coupling a b^T.")

**Read the output.** The independent coupling carries strictly more entropy than the sharp one — it is the spread-out extreme the bonus favours. So $\varepsilon$ is an *exchange rate*: at small $\varepsilon$ the transport cost wins and the plan stays sharp (the LP optimum); at large $\varepsilon$ the entropy bonus wins and the plan relaxes toward $a\,b^\top$. The next notebook computes these plans for real and watches the diagonal melt as $\varepsilon$ grows.

## Your turn

1. **Entropy is concave.** Take two couplings $P, Q \in T(a, b)$ and their midpoint $\tfrac{1}{2}(P + Q)$. Check that $H\big(\tfrac{1}{2}(P+Q)\big) \ge \tfrac{1}{2}\big(H(P) + H(Q)\big)$. This concavity is exactly why the regularised objective has a unique minimum.
2. **The two limits, numerically.** Build `gibbs_kernel(C, eps)` for a tiny $\varepsilon$ and a huge one. Confirm it approaches the indicator of $C = 0$ in one limit and the all-ones matrix in the other.
3. **Not every $u, v$ works.** Pick random positive vectors $u, v$ and form $P = \mathrm{diag}(u)\,K\,\mathrm{diag}(v)$. Does it satisfy the marginals $P\mathbf{1} = a$, $P^\top\mathbf{1} = b$? Finding the $u, v$ that *do* is precisely the job of the Sinkhorn algorithm.

## What you built

- You wrote the entropic OT objective and saw why the entropy bonus makes it strongly convex (a unique, smooth solution).
- You built the Gibbs kernel $K = e^{-C/\varepsilon}$ and watched $\varepsilon$ tune it from a sharp diagonal to a flat sheet.
- You predicted the two limits — $\varepsilon \to 0$ to the LP, $\varepsilon \to \infty$ to the independent coupling.

You now hold the object every fast OT solver is built on. The blur you introduced on purpose is what makes the next step possible.

## What's next

In `10_sinkhorn_algorithm` we find the scalings $u, v$ that bend the Gibbs kernel onto the marginal constraints — the five-line Sinkhorn iteration — and watch it converge geometrically while the $\varepsilon$ dial trades sharpness for speed.

## References

- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *NeurIPS* (2013).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 4. DOI:10.1561/2200000073
- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016), sec. 7.5.

**Previous:** `notebooks/03_ClassicalOptimalTransport/08_kantorovich_duality.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/10_sinkhorn_algorithm.ipynb`